# Análisis de datasets

Notebook sencillo para revisar `Sentences_AllAgree.csv` y `FinTextSen.csv` antes de ejecutar los experimentos de sentimiento financiero.

In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path('..')

datasets = {
    'Sentences_AllAgree': ROOT / 'Sentences_AllAgree.csv',
    'FinTextSen': ROOT / 'FinTextSen.csv',
}

label_map = {
    0: 'negative',
    1: 'neutral',
    2: 'positive',
}

In [2]:
def analizar_dataset(nombre, ruta):
    df = pd.read_csv(ruta)
    textos = df['sentences'].fillna('').astype(str).str.strip()
    etiquetas = df['labels']

    palabras = textos.apply(lambda texto: len(texto.split()) if texto else 0)
    caracteres = textos.str.len()

    resumen = pd.DataFrame({
        'dataset': [nombre],
        'registros': [len(df)],
        'columnas': [df.shape[1]],
        'textos_vacios': [textos.eq('').sum()],
        'etiquetas_vacias': [etiquetas.isna().sum()],
        'promedio_palabras': [round(palabras.mean(), 2)],
        'max_palabras': [palabras.max()],
        'promedio_caracteres': [round(caracteres.mean(), 2)],
        'duplicados_sentences': [df.duplicated(subset=['sentences']).sum()],
        'duplicados_fila_completa': [df.duplicated().sum()],
    })

    clases = (
        df['labels']
        .value_counts(dropna=False)
        .rename_axis('label')
        .reset_index(name='registros')
        .sort_values('label')
    )
    clases['sentimiento'] = clases['label'].map(label_map)
    clases['porcentaje'] = (clases['registros'] / len(df) * 100).round(2)
    clases = clases[['label', 'sentimiento', 'registros', 'porcentaje']]

    return df, resumen, clases

## Resumen general

In [3]:
resultados = {}
resumenes = []

for nombre, ruta in datasets.items():
    df, resumen, clases = analizar_dataset(nombre, ruta)
    resultados[nombre] = {'datos': df, 'resumen': resumen, 'clases': clases}
    resumenes.append(resumen)

resumen_general = pd.concat(resumenes, ignore_index=True)
resumen_general

,dataset,registros,columnas,textos_vacios,etiquetas_vacias,promedio_palabras,max_palabras,promedio_caracteres,duplicados_sentences,duplicados_fila_completa
0,Sentences_AllAgree,2264,2,0,0,22.44,81,121.96,5,5
1,FinTextSen,1700,2,20,0,6.12,25,34.61,448,439


## Porcentaje de clases

In [4]:
porcentajes_clases = pd.concat(
    [
        resultados[nombre]['clases'].assign(dataset=nombre)
        for nombre in datasets
    ],
    ignore_index=True
)

porcentajes_clases[['dataset', 'label', 'sentimiento', 'registros', 'porcentaje']]

,dataset,label,sentimiento,registros,porcentaje
0,Sentences_AllAgree,0,negative,303,13.38
1,Sentences_AllAgree,1,neutral,1391,61.44
2,Sentences_AllAgree,2,positive,570,25.18
3,FinTextSen,0,negative,581,34.18
4,FinTextSen,1,neutral,27,1.59
5,FinTextSen,2,positive,1092,64.24


## Sentences_AllAgree.csv

In [5]:
resultados['Sentences_AllAgree']['resumen']

,dataset,registros,columnas,textos_vacios,etiquetas_vacias,promedio_palabras,max_palabras,promedio_caracteres,duplicados_sentences,duplicados_fila_completa
0,Sentences_AllAgree,2264,2,0,0,22.44,81,121.96,5,5


In [6]:
resultados['Sentences_AllAgree']['clases']

,label,sentimiento,registros,porcentaje
2,0,negative,303,13.38
0,1,neutral,1391,61.44
1,2,positive,570,25.18


## FinTextSen.csv

In [7]:
resultados['FinTextSen']['resumen']

,dataset,registros,columnas,textos_vacios,etiquetas_vacias,promedio_palabras,max_palabras,promedio_caracteres,duplicados_sentences,duplicados_fila_completa
0,FinTextSen,1700,2,20,0,6.12,25,34.61,448,439


In [8]:
resultados['FinTextSen']['clases']

,label,sentimiento,registros,porcentaje
1,0,negative,581,34.18
2,1,neutral,27,1.59
0,2,positive,1092,64.24


## Limpieza Mínima

`FinTextSen.csv` contiene textos vacíos. Para inferencia o entrenamiento conviene eliminarlos antes de pasar los textos al modelo.

In [9]:
fintextsen = resultados['FinTextSen']['datos'].copy()
fintextsen['sentences'] = fintextsen['sentences'].fillna('').astype(str).str.strip()

fintextsen_limpio = fintextsen[fintextsen['sentences'] != ''].copy()

print('Registros originales:', len(fintextsen))
print('Registros sin textos vacíos:', len(fintextsen_limpio))
print('Textos eliminados:', len(fintextsen) - len(fintextsen_limpio))

Registros originales: 1700
Registros sin textos vacíos: 1680
Textos eliminados: 20


## Exportar FinTextSen limpio

In [ ]:
ruta_salida = ROOT / 'FinTextSen_clean.csv'
fintextsen_limpio.to_csv(ruta_salida, index=False)

print(f'Archivo generado: {ruta_salida}')
print('Registros guardados:', len(fintextsen_limpio))